# Notebook 09: AI Driver

## Before You Start
> **If anything behaves unexpectedly, restart the kernel first: Kernel menu → Restart Kernel and Clear All Outputs. Then run the cells from the top.**
>
> **Make sure Notebook 08 is working cleanly before starting this one.**

## ADAS Connection
This is it. Everything you have built this week comes together in this notebook:

- **Notebook 01-03:** You learned what the robot can do -- LEDs, motors, servos
- **Notebook 04-05:** You gave it eyes -- camera, color detection
- **Notebook 06:** You gave it reflexes -- color following
- **Notebook 07-08:** You gave it a brain -- AI model, vision to decision
- **Notebook 09:** You put it all together -- a fully autonomous AI driver

The pipeline your robot runs in this notebook is the same fundamental architecture used in real autonomous vehicles:

```
CAMERA → DETECT → OBSERVE → AI DECIDES → MOTORS ACT → repeat
```

Tesla calls this loop their **autonomy stack**. You just built a miniature version of it.

---

## How It Works
The AI Driver loop runs continuously:

1. Capture a frame from the camera
2. Detect the target color and any obstacles
3. Build a natural language observation
4. Send the observation to the AI model
5. Parse the decision: FORWARD, LEFT, RIGHT, or STOP
6. Execute the decision with the motors
7. Display everything on screen
8. Repeat

The AI runs every N frames -- not every single frame -- because the model takes about 1 second to respond. Between AI calls the robot uses its last decision to keep moving smoothly.

---

## The Code
Run this cell to set up everything.

In [1]:
import cv2
import numpy as np
import ipywidgets as widgets
import threading
import time
import sys
from IPython.display import display
import RPi.GPIO as GPIO


sys.path.insert(0, '/home/pi/lab')
import motors
import ai_driver


def bgr8_to_jpeg(frame):
    return bytes(cv2.imencode('.jpg', frame)[1])

# Open camera
try:
    cap.release()
except:
    pass

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH,  640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M','J','P','G'))
cap.set(cv2.CAP_PROP_BRIGHTNESS, 40)
cap.set(cv2.CAP_PROP_CONTRAST,   40)

# Color ranges
COLOR_RANGES = {
    'red':    ([0,   43,  46],  [10,  255, 255]),
    'green':  ([35,  43,  46],  [77,  255, 255]),
    'blue':   ([100, 43,  46],  [124, 255, 255]),
    'yellow': ([26,  43,  46],  [34,  255, 255]),
    'orange': ([11,  43,  46],  [25,  255, 255]),
}

# Setup motors
motors.setup()

# Helper functions for pan-tilt
def angle_to_duty(angle):
    return 2.5 + (angle / 180.0) * 10.0


# Warm up camera
print('Warming up camera...')
for _ in range(20):
    cap.read()
    time.sleep(0.05)

# Warm up AI model
print('Warming up AI model...')
ret = ai_driver.ask('hello')

print('Camera ready!')
print('Motors ready!')
if "error" in ret.lower():
    print(f'AI model not ready. Check config: {ret}')
else:
    print('AI model ready!')
print()
print('>>> Robot must be on the floor with space to move before running the driver.')

Warming up camera...
Warming up AI model...
Camera ready!
Motors ready!
AI model ready!

>>> Robot must be on the floor with space to move before running the driver.


---

## YOUR TURN -- Tweak Zone 1: Configure Your AI Driver

These are the parameters that define your robot's personality and behavior. Read each one carefully. The decisions you make here will determine how your robot performs on the obstacle course.

**This is your team's configuration. Own it.**

> **Think like an engineer:** Real autonomous vehicle teams spend months tuning these parameters in simulation before testing on a real car. You have a few hours. What is your strategy?

In [4]:
# ═══════════════════════════════════════════════════════════
#   YOUR TEAM CONFIGURATION -- tune these for the race
# ═══════════════════════════════════════════════════════════

# -- Vision --
TARGET_COLOR = 'green'       # your team color
MIN_RADIUS   = 20          # minimum blob size to track
DEAD_ZONE    = 50          # pixels from center before turning

# -- Speed --
FOLLOW_SPEED = 35          # forward speed when target is centered (0-100)
TURN_SPEED   = 30          # turning speed when target is off-center (0-100)
STOP_DISTANCE = 25         # cm -- stop if obstacle closer than this

# -- AI --
AI_EVERY_N_FRAMES = 10     # how often to ask the AI (every N frames)
                           # lower = more AI decisions, slower loop
                           # higher = fewer AI decisions, faster loop

SYSTEM_PROMPT = """You are the AI brain of a small robot car.
Your job is to follow a colored target while avoiding obstacles.
If an obstacle is closer than 25cm, safety is the priority -- STOP or turn away.
Otherwise follow the target.
Respond with ONLY one word: FORWARD, LEFT, RIGHT, or STOP.
No explanation. No punctuation. Just the single word."""

# ═══════════════════════════════════════════════════════════

color_lower  = np.array(COLOR_RANGES[TARGET_COLOR][0])
color_upper  = np.array(COLOR_RANGES[TARGET_COLOR][1])
FRAME_CENTER = 320

print('Configuration:')
print(f'  Target color:     {TARGET_COLOR}')
print(f'  Follow speed:     {FOLLOW_SPEED}')
print(f'  Turn speed:       {TURN_SPEED}')
print(f'  Dead zone:        +/- {DEAD_ZONE}px from center')
print(f'  Stop distance:    {STOP_DISTANCE}cm')
print(f'  AI every:         {AI_EVERY_N_FRAMES} frames')

def decide_debug(observation, system_prompt):
    raw = ai_driver.ask(f"{system_prompt}\n\nObservation: {observation}")
    print(f"Raw AI response: '{raw}'")
    raw_upper = raw.strip().upper()
    for word in raw_upper.split():
        if word in ['FORWARD', 'LEFT', 'RIGHT', 'STOP']:
            print(f"Parsed decision: {word}")
            return word
    print(f"No valid command found -- defaulting to STOP")
    return 'STOP'

Configuration:
  Target color:     red
  Follow speed:     35
  Turn speed:       30
  Dead zone:        +/- 50px from center
  Stop distance:    25cm
  AI every:         10 frames


---

## YOUR TURN -- Tweak Zone 2: Start the AI Driver

Place the robot on the floor. Hold your colored target in front of the camera.

Run the START cell. The robot will:
- Follow your colored target using computer vision
- Ask the AI for a decision every N frames
- Display what it sees, what it observes, and what it decided
- Stop safely if an obstacle is detected

Run the STOP cell when you are done.

> **Safety:** Keep hands near the robot. Start with low speeds until you trust the behavior.
>
> **Watch the status display:** It shows the current observation and AI decision in real time. This is your window into what the robot is thinking.

In [5]:
# START CELL
feed_widget   = widgets.Image(format='jpeg', width=640, height=480)
obs_widget    = widgets.Label(value='Observation: starting...')
dec_widget    = widgets.Label(value='Decision: --')
status_widget = widgets.Label(value='Status: --')
display(obs_widget, dec_widget, status_widget, feed_widget)

running      = True
frame_count  = 0
last_decision = 'STOP'
last_obs      = 'Starting...'


def build_observation(frame, distance_cm=200):
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, color_lower, color_upper)
    mask = cv2.erode(mask,  None, iterations=2)
    mask = cv2.dilate(mask, None, iterations=2)
    mask = cv2.GaussianBlur(mask, (3,3), 0)
    cnts = cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    
    cx, cy, radius = None, None, 0
    if len(cnts) > 0:
        cnt = max(cnts, key=cv2.contourArea)
        (cx, cy), radius = cv2.minEnclosingCircle(cnt)
        if radius < MIN_RADIUS:
            cx, cy, radius = None, None, 0
    
    # Build color observation
    if cx is not None:
        if cx < FRAME_CENTER - DEAD_ZONE:
            position = "left side"
        elif cx > FRAME_CENTER + DEAD_ZONE:
            position = "right side"
        else:
            position = "center"
        
        if radius < 30:
            dist_desc = "far away"
        elif radius < 80:
            dist_desc = "medium distance"
        else:
            dist_desc = "very close"
        
        color_obs = f"Target '{TARGET_COLOR}' on the {position} at X={int(cx)}. {dist_desc}."
    else:
        color_obs = "No target detected."
        cx, cy = None, None
    
    # Build obstacle observation
    if distance_cm < STOP_DISTANCE:
        obstacle_obs = f"WARNING: Obstacle {distance_cm}cm ahead."
    elif distance_cm < 50:
        obstacle_obs = f"Obstacle {distance_cm}cm ahead, caution."
    else:
        obstacle_obs = "Path clear."
    
    return f"{color_obs} {obstacle_obs}", cx, cy, int(radius)

def execute_decision(decision, cx):
    """Execute a driving decision from the AI."""
    if decision == 'FORWARD':
        motors.forward(FOLLOW_SPEED)
    elif decision == 'LEFT':
        motors.turn_left(TURN_SPEED)
    elif decision == 'RIGHT':
        motors.turn_right(TURN_SPEED)
    else:
        motors.brake()
        
# Pan/tilt state for search behavior
pan_angle = 90
pan_direction = 1  # 1 = right, -1 = left
SEARCH_STEP = 15   # degrees per search step


def ai_driver_loop():
    global running, frame_count, last_decision, last_obs, pan_angle, pan_direction
    
    test_obs = "Target 'red' on the left side at X=189. medium distance. Path clear."
    decide_debug(test_obs, SYSTEM_PROMPT)

    while running:
        ret, frame = cap.read()
        if not ret:
            break
        
        frame_count += 1
        
        # Build observation every frame
        obs, cx, cy, radius = build_observation(frame)
        
        # Pan/tilt behavior
        if cx is not None:
            # Target detected -- pan to follow it
            # Map X (0-640) to pan angle (160-20) -- inverted because panning right moves target left
            target_pan = 160 - int((cx / 640) * 140)
            pan_to(target_pan, delay=0.05)
        else:
            # Target lost -- search pattern
            pan_angle += SEARCH_STEP * pan_direction
            if pan_angle >= 160:
                pan_angle = 160
                pan_direction = -1
            elif pan_angle <= 20:
                pan_angle = 20
                pan_direction = 1
            pan_to(pan_angle, delay=0.1)
        
        # Ask AI every N frames
        if frame_count % AI_EVERY_N_FRAMES == 0:
            last_obs = obs
            last_decision = ai_driver.decide(obs, SYSTEM_PROMPT)
        
        # Execute last known decision
        execute_decision(last_decision, cx)
        
        # Draw on frame
        cv2.line(frame, (FRAME_CENTER, 0), (FRAME_CENTER, 480), (255,255,255), 1)
        cv2.line(frame, (FRAME_CENTER-DEAD_ZONE, 0), (FRAME_CENTER-DEAD_ZONE, 480), (0,255,0), 1)
        cv2.line(frame, (FRAME_CENTER+DEAD_ZONE, 0), (FRAME_CENTER+DEAD_ZONE, 480), (0,255,0), 1)
        
        if cx is not None:
            cv2.circle(frame, (int(cx), int(cy)), radius, (0,255,255), 2)
            cv2.circle(frame, (int(cx), int(cy)), 5, (0,255,255), -1)
        
        color = (0,255,0) if last_decision == 'FORWARD' else (0,165,255) if last_decision in ['LEFT','RIGHT'] else (0,0,255)
        cv2.putText(frame, f'AI: {last_decision}', (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
        cv2.putText(frame, f'Frame: {frame_count}', (10, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        
        # Update widgets
        obs_widget.value    = f'Observation: {last_obs[:80]}...'
        dec_widget.value    = f'Decision: {last_decision}'
        status_widget.value = f'Frame: {frame_count}  |  AI calls: {frame_count // AI_EVERY_N_FRAMES}'
        feed_widget.value   = bgr8_to_jpeg(frame)
        
        time.sleep(0.033)
    
    motors.brake()
    pan_to(90, 0.5)   # return to center on stop


driver_thread = threading.Thread(target=ai_driver_loop)
driver_thread.daemon = True
driver_thread.start()
print('AI Driver started. Run the STOP cell to stop the robot.')

Label(value='Observation: starting...')

Label(value='Decision: --')

Label(value='Status: --')

Image(value=b'', format='jpeg', height='480', width='640')

AI Driver started. Run the STOP cell to stop the robot.
Raw AI response: 'Right'
Parsed decision: RIGHT


In [6]:
# STOP CELL -- run this to stop the robot
running = False
time.sleep(0.5)
motors.brake()
pan_to(90, 0.5)
print('AI Driver stopped. Camera centered.')

AI Driver stopped. Camera centered.


---

## YOUR TURN -- Tweak Zone 3: Tune for the Race

Go back to Tweak Zone 1 and experiment. The goal is to find the combination that navigates the obstacle course fastest with the fewest penalties.

Use this tuning guide:

| Problem | Try this |
|---------|----------|
| Robot loses target too easily | Decrease MIN_RADIUS |
| Robot reacts to false detections | Increase MIN_RADIUS |
| Robot turns too aggressively | Decrease TURN_SPEED or increase DEAD_ZONE |
| Robot misses turns | Increase TURN_SPEED or decrease DEAD_ZONE |
| Robot drives too fast and crashes | Decrease FOLLOW_SPEED |
| Robot is too slow on straightaways | Increase FOLLOW_SPEED |
| Robot stops too far from obstacles | Decrease STOP_DISTANCE |
| Robot stops too late | Increase STOP_DISTANCE |
| AI decisions feel slow | Increase AI_EVERY_N_FRAMES |
| AI decisions feel wrong | Improve SYSTEM_PROMPT |

> **Team strategy:** Assign one person to track every parameter change and the result. Change ONE thing at a time. This is how real engineering teams tune systems.

---

## What Happened?

Think about these questions with your team:

1. What was the biggest difference between AI-driven behavior (Notebook 09) and rule-based behavior (Notebook 06)?
2. What parameter had the biggest impact on performance?
3. How would you improve the SYSTEM_PROMPT to handle the obstacle course better?
4. Real self-driving cars run this loop at 1000+ frames per second. Your robot runs it at about 30 frames but only asks the AI every 10. What are the tradeoffs?

---

## CHALLENGE -- Advanced Students

The current AI driver has no memory -- every decision is made from a single frame with no knowledge of what happened before.

Add a **decision history** to the observation. Include the last 3 decisions and their outcomes. Does giving the AI context about what it just did improve its decision making?

Hint: keep a list of recent decisions and append them to the observation string.

In [ ]:
# YOUR CODE HERE
# ═══════════════════════════════════════
HISTORY_LENGTH = 3    # how many past decisions to include
# ═══════════════════════════════════════

decision_history = []

def build_observation_with_history(frame, distance_cm=200):
    global decision_history
    # build the base observation
    # append recent decision history
    # return enhanced observation
    pass


---

## Always clean up when you are done!

In [ ]:
running = False
time.sleep(0.5)
motors.cleanup()
cap.release()
print('AI Driver shut down. Motors and camera released.')